<a href="https://colab.research.google.com/github/shahd-w21/FlyRank/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahd-w21/FlyRank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm choosing Lane 4: CTR / Engagement Opportunity Scoring.
In Week 1, I discovered that `comparison article` pages have a lower click-through rate
than other content types even when they rank at the exact same position tier — meaning
position alone doesn't explain how many clicks a page gets. That's the core mechanic this
lane is built around: comparing each page's CTR to what's expected for its position tier,
and flagging pages that fall short. I want to turn that Week 1 observation into an actual
ranked list of pages worth reviewing.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Decision this improves:which visible pages (real traffic, real ranking position) are
under-capturing clicks relative to other pages at the same position, and should be reviewed
first.

Who acts on it:a content reviewer, who checks the page's title, meta description, and
whether it actually matches what the searcher wants — then rewrites it if needed.

Cost of a wrong call:
- False positive (flagged, but page was actually fine) → reviewer wastes time rewriting
  something that didn't need it.
- False negative (missed, but page was really underperforming) → the page keeps losing
  clicks it could be getting, with nobody looking at it.

Why this needs data, not just eyeballing:with tens of thousands of pages, no one can
manually check what "normal" CTR looks like at every position tier and content type — you
need the data to establish the baseline before you can spot who's below it.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [13]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [14]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
visible = df[df["impressions_90d"] >= 100]
print(f"Pages with enough traffic to trust CTR: {visible.shape[0]} of {df.shape[0]} total")

ctr_by_type_tier = (
    visible.groupby(["position_tier", "content_type"])["ctr"]
    .mean()
    .round(4)
    .unstack()
)
print(ctr_by_type_tier)

# size of the gap at one tier, as a concrete number
gap = ctr_by_type_tier.loc["page_1", "feedly article"] - ctr_by_type_tier.loc["page_1", "comparison article"]
print(f"\nAt page_1, feedly article beats comparison article by {gap:.4f} CTR "
      f"({ctr_by_type_tier.loc['page_1','feedly article']/ctr_by_type_tier.loc['page_1','comparison article']:.1f}x)")

Pages with enough traffic to trust CTR: 22006 of 30000 total
content_type   comparison article  feedly article  keyword article
position_tier                                                     
deep                          NaN          0.0440           0.0555
page_1                     0.1412          0.9048           0.3458
page_3_5                   0.0940          0.1656           0.1427
striking                   0.1474          0.3580           0.2559
top_3                      0.0000          2.9000           0.3104

At page_1, feedly article beats comparison article by 0.7636 CTR (6.4x)


Of the 30,000 pages in this dataset, 22,006 have enough traffic (impressions_90d >= 100)
to trust their CTR. Even restricted to that filtered set, 'comparison articl' pages get roughly
6.4x fewer clicks than 'feedly article' pages at the identical 'page_1' rank — a real,
measurable gap that has nothing to do with position. That gap is exactly the kind of signal
this lane is meant to catch and rank.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

Can say: which pages are observed to sit below their position tier's average CTR, and
by how much a directional signal for prioritizing review.

Cannot say: why any specific page underperforms, that rewriting it will fix the CTR,
or anything about how Google's ranking or CTR algorithm actually works. Any output from
this lane is decision-support for a human reviewer, not a guarantee.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.